In [1]:
# cell 1
# Install lightweight sparse/index dependencies.

!pip -q install -U scipy joblib tqdm psutil numpy

import os
import re
import json
import html
import unicodedata
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import scipy.sparse as sp
import joblib
import psutil
from tqdm.auto import tqdm

print("CPU count:", os.cpu_count())
print("RAM GB:", round(psutil.virtual_memory().total / (1024 ** 3), 2))

CPU count: 8
RAM GB: 50.99


In [2]:
# cell 2
# Mount Google Drive and define paths.

from google.colab import drive
drive.mount("/content/drive")

DATASET = "hotpotqa"

PROJECT_DIR = Path("/content/drive/MyDrive/final_project/idea_1")
DATASET_DIR = PROJECT_DIR / "kg" / DATASET
CONSTRUCT_DIR = DATASET_DIR / "kg_construct"

ENTITY_CATALOG_JSONL_PATH = CONSTRUCT_DIR / "hotpotqa_entities.jsonl"
KG_META_JSON_PATH = CONSTRUCT_DIR / "hotpotqa_hetero_kg_meta.json"

# New BM25 index outputs.
TOKEN_BM25_INDEX_PATH = CONSTRUCT_DIR / "hotpotqa_entity_token_bm25.joblib"
CHAR3_BM25_INDEX_PATH = CONSTRUCT_DIR / "hotpotqa_entity_char3_bm25.joblib"
BM25_META_JSON_PATH = CONSTRUCT_DIR / "hotpotqa_entity_bm25_indexes_meta.json"

required_paths = [
    ENTITY_CATALOG_JSONL_PATH,
    KG_META_JSON_PATH,
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

print("Construct dir:", CONSTRUCT_DIR)
print("Entity catalog:", ENTITY_CATALOG_JSONL_PATH)

Mounted at /content/drive
Construct dir: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct
Entity catalog: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entities.jsonl


In [3]:
# cell 3
# Define safe writing and normalization helpers.

def atomic_json_dump(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + ".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp_path, path)


def normalize_entity_text(text):
    # Match KG entity normalization.
    if text is None:
        return ""

    text = str(text)
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)

    replacements = {
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2013": "-",
        "\u2014": "-",
        "\u2212": "-",
        "\u00a0": " ",
    }

    for src, dst in replacements.items():
        text = text.replace(src, dst)

    text = re.sub(r"\s+", " ", text).strip()
    text = text.strip(" \t\r\n\"'`")
    text = text.lower()

    return text


def file_size_mb(path):
    path = Path(path)
    return path.stat().st_size / (1024 ** 2) if path.exists() else None


def print_mem(prefix="Memory"):
    mem = psutil.virtual_memory()
    print(
        f"{prefix}: used={round(mem.used / (1024 ** 3), 2)} GB, "
        f"available={round(mem.available / (1024 ** 3), 2)} GB"
    )

In [4]:
#cell 4
# Define token and char 3-gram analyzers.

TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?")

def token_analyzer(text):
    # Token BM25 analyzer.
    text = normalize_entity_text(text)
    return TOKEN_RE.findall(text)


def char3_analyzer(text):
    # Character 3-gram BM25 analyzer.
    text = normalize_entity_text(text)

    if not text:
        return []

    # Keep spaces because entity phrase structure is useful.
    padded = f"  {text}  "

    if len(padded) < 3:
        return [padded]

    return [padded[i:i + 3] for i in range(len(padded) - 2)]

In [5]:
#cell 5
# Load unique KG entities.

entity_texts = []
bad_rows = []

with open(ENTITY_CATALOG_JSONL_PATH, "r", encoding="utf-8") as f:
    for row, line in enumerate(tqdm(f, desc="Loading entities")):
        if not line.strip():
            continue

        rec = json.loads(line)
        entity_id = rec.get("entity_id")
        entity = rec.get("entity")

        if entity_id != row:
            bad_rows.append({
                "row": row,
                "entity_id": entity_id,
                "entity": entity,
                "reason": "entity_id does not match row",
            })
            continue

        if not isinstance(entity, str) or not entity:
            bad_rows.append({
                "row": row,
                "entity_id": entity_id,
                "entity": entity,
                "reason": "invalid entity text",
            })
            continue

        # Store normalized text. Row index is entity_id.
        entity_texts.append(normalize_entity_text(entity))

if bad_rows:
    print(json.dumps(bad_rows[:20], ensure_ascii=False, indent=2))
    raise RuntimeError(f"Bad entity rows found: {len(bad_rows)}")

num_entities = len(entity_texts)

print("Total entities:", num_entities)
print("First 5 entities:")
for i in range(5):
    print(i, entity_texts[i])

print_mem("After loading entities")

Loading entities: 0it [00:00, ?it/s]

Total entities: 327873
First 5 entities:
0 meet corliss archer
1 january 7, 1943
2 september 30, 1956
3 cbs
4 a date with judy
After loading entities: used=1.65 GB, available=49.34 GB


In [6]:
# cell 6
# Build a BM25 sparse index.

def build_bm25_index(
    texts,
    analyzer,
    index_type,
    k1=1.5,
    b=0.75,
    min_df=1,
):
    """
    Build BM25 document-term matrix.
    Rows are entity_id.
    Columns are lexical terms.
    """

    vocab = {}
    indptr = [0]
    indices = []
    data = []

    df_counter = Counter()
    doc_lens = np.zeros(len(texts), dtype=np.float32)

    for text in tqdm(texts, desc=f"Counting terms for {index_type}"):
        terms = analyzer(text)
        counts = Counter(terms)

        if min_df <= 1:
            filtered_counts = counts
        else:
            # min_df filtering is applied later globally.
            filtered_counts = counts

        doc_lens[len(indptr) - 1] = sum(filtered_counts.values())

        for term, tf in filtered_counts.items():
            term_id = vocab.get(term)
            if term_id is None:
                term_id = len(vocab)
                vocab[term] = term_id

            indices.append(term_id)
            data.append(float(tf))
            df_counter[term_id] += 1

        indptr.append(len(indices))

    n_docs = len(texts)
    n_terms = len(vocab)

    print(f"{index_type} vocab size:", n_terms)
    print(f"{index_type} nonzero raw tf:", len(data))

    X = sp.csr_matrix(
        (
            np.asarray(data, dtype=np.float32),
            np.asarray(indices, dtype=np.int32),
            np.asarray(indptr, dtype=np.int64),
        ),
        shape=(n_docs, n_terms),
        dtype=np.float32,
    )

    X.sort_indices()

    avgdl = float(doc_lens.mean())
    if avgdl <= 0:
        raise RuntimeError(f"Invalid avgdl for {index_type}: {avgdl}")

    # Compute df aligned by term_id.
    df = np.zeros(n_terms, dtype=np.float32)
    for term_id, value in df_counter.items():
        df[term_id] = value

    # BM25 idf.
    idf = np.log(1.0 + ((n_docs - df + 0.5) / (df + 0.5))).astype(np.float32)

    # Convert TF matrix to BM25 matrix.
    X = X.tocoo(copy=False)

    denom = X.data + k1 * (1.0 - b + b * (doc_lens[X.row] / avgdl))
    X.data = (X.data * (k1 + 1.0) / denom) * idf[X.col]

    bm25_matrix = X.tocsr()
    bm25_matrix.sort_indices()

    index_payload = {
        "version": "entity_bm25_sparse_v1",
        "dataset": DATASET,
        "index_type": index_type,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "k1": float(k1),
        "b": float(b),
        "num_entities": int(n_docs),
        "num_terms": int(n_terms),
        "avgdl": float(avgdl),
        "entity_id_is_row": True,
        "entities": texts,
        "vocab": vocab,
        "idf": idf,
        "doc_lens": doc_lens,
        "bm25_matrix": bm25_matrix,
        "normalization": "same_as_KG_normalize_entity_text",
    }

    return index_payload

In [7]:
#cell 7
# Build token BM25 index.

token_bm25_index = build_bm25_index(
    texts=entity_texts,
    analyzer=token_analyzer,
    index_type="token_bm25",
    k1=1.5,
    b=0.75,
)

print_mem("After token BM25 build")
print("Token BM25 matrix shape:", token_bm25_index["bm25_matrix"].shape)
print("Token BM25 nnz:", token_bm25_index["bm25_matrix"].nnz)

Counting terms for token_bm25:   0%|          | 0/327873 [00:00<?, ?it/s]

token_bm25 vocab size: 124503
token_bm25 nonzero raw tf: 842075
After token BM25 build: used=1.84 GB, available=49.15 GB
Token BM25 matrix shape: (327873, 124503)
Token BM25 nnz: 842075


In [8]:
#cell 8
# Save token BM25 index.

joblib.dump(
    token_bm25_index,
    TOKEN_BM25_INDEX_PATH,
    compress=3,
)

print("Saved token BM25:", TOKEN_BM25_INDEX_PATH)
print("Size MB:", round(file_size_mb(TOKEN_BM25_INDEX_PATH), 2))

Saved token BM25: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_token_bm25.joblib
Size MB: 7.73


In [9]:
#cell 9
# Build char 3-gram BM25 index.

char3_bm25_index = build_bm25_index(
    texts=entity_texts,
    analyzer=char3_analyzer,
    index_type="char3_bm25",
    k1=1.5,
    b=0.75,
)

print_mem("After char3 BM25 build")
print("Char3 BM25 matrix shape:", char3_bm25_index["bm25_matrix"].shape)
print("Char3 BM25 nnz:", char3_bm25_index["bm25_matrix"].nnz)

Counting terms for char3_bm25:   0%|          | 0/327873 [00:00<?, ?it/s]

char3_bm25 vocab size: 37385
char3_bm25 nonzero raw tf: 5821630
After char3 BM25 build: used=2.17 GB, available=48.82 GB
Char3 BM25 matrix shape: (327873, 37385)
Char3 BM25 nnz: 5821630


In [10]:
#cell 10
# Save char 3-gram BM25 index.

joblib.dump(
    char3_bm25_index,
    CHAR3_BM25_INDEX_PATH,
    compress=3,
)

print("Saved char3 BM25:", CHAR3_BM25_INDEX_PATH)
print("Size MB:", round(file_size_mb(CHAR3_BM25_INDEX_PATH), 2))

Saved char3 BM25: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_char3_bm25.joblib
Size MB: 30.71


In [11]:
#cell 11
# Define BM25 search helpers.

def get_query_terms(query, index_type):
    if index_type == "token_bm25":
        return token_analyzer(query)

    if index_type == "char3_bm25":
        return char3_analyzer(query)

    raise ValueError(f"Unknown index_type: {index_type}")


def search_entity_bm25(query, index_payload, top_k=10):
    # Search entity BM25 index.
    vocab = index_payload["vocab"]
    matrix = index_payload["bm25_matrix"]
    entities = index_payload["entities"]
    index_type = index_payload["index_type"]

    terms = get_query_terms(query, index_type)
    term_ids = sorted({vocab[t] for t in terms if t in vocab})

    if not term_ids:
        return []

    scores = np.asarray(matrix[:, term_ids].sum(axis=1)).ravel()

    top_k = min(int(top_k), len(scores))
    if top_k <= 0:
        return []

    candidate_ids = np.argpartition(-scores, top_k - 1)[:top_k]
    candidate_ids = candidate_ids[np.argsort(-scores[candidate_ids])]

    results = []
    for entity_id in candidate_ids:
        score = float(scores[entity_id])
        if score <= 0:
            continue

        results.append({
            "entity_id": int(entity_id),
            "entity": entities[int(entity_id)],
            "score": score,
        })

    return results


def print_results(title, results):
    print("\n" + title)
    for rank, item in enumerate(results, start=1):
        print(
            f"{rank:02d}. entity_id={item['entity_id']} "
            f"score={item['score']:.4f} entity={item['entity']}"
        )

In [12]:
#cell 12
# Reload indexes and run sanity checks.

loaded_token_index = joblib.load(TOKEN_BM25_INDEX_PATH)
loaded_char3_index = joblib.load(CHAR3_BM25_INDEX_PATH)

test_queries = [
    "german language",
    "meet corliss archer",
    "high german consonant shift",
    "Barack Obama",
]

for query in test_queries:
    token_results = search_entity_bm25(query, loaded_token_index, top_k=5)
    char3_results = search_entity_bm25(query, loaded_char3_index, top_k=5)

    print_results(f"TOKEN BM25 query={query!r}", token_results)
    print_results(f"CHAR3 BM25 query={query!r}", char3_results)


TOKEN BM25 query='german language'
01. entity_id=63882 score=15.5667 entity=german language
02. entity_id=64254 score=13.0384 entity=german language learners
03. entity_id=290828 score=13.0384 entity=german language countries
04. entity_id=258512 score=13.0384 entity=german language expansion
05. entity_id=38278 score=10.2663 entity=language

CHAR3 BM25 query='german language'
01. entity_id=63882 score=77.6276 entity=german language
02. entity_id=63843 score=66.1351 entity=germanic language
03. entity_id=64254 score=61.2509 entity=german language learners
04. entity_id=258512 score=59.9988 entity=german language expansion
05. entity_id=290828 score=59.9988 entity=german language countries

TOKEN BM25 query='meet corliss archer'
01. entity_id=0 score=27.1981 entity=meet corliss archer
02. entity_id=353 score=22.3403 entity=corliss archer
03. entity_id=9 score=15.1826 entity=corliss
04. entity_id=62265 score=12.5321 entity=archer
05. entity_id=48525 score=12.2384 entity=richard corliss


In [13]:
#cell 13
# Save BM25 metadata.

bm25_meta = {
    "dataset": DATASET,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "entity_id_mapping": "entity_id equals BM25 row index and KG entity node id",
    "indexes": {
        "entity_token_bm25": {
            "path": str(TOKEN_BM25_INDEX_PATH),
            "format": "joblib",
            "index_type": "token_bm25",
            "num_entities": int(loaded_token_index["num_entities"]),
            "num_terms": int(loaded_token_index["num_terms"]),
            "matrix_shape": list(loaded_token_index["bm25_matrix"].shape),
            "matrix_nnz": int(loaded_token_index["bm25_matrix"].nnz),
            "k1": loaded_token_index["k1"],
            "b": loaded_token_index["b"],
        },
        "entity_char3_bm25": {
            "path": str(CHAR3_BM25_INDEX_PATH),
            "format": "joblib",
            "index_type": "char3_bm25",
            "num_entities": int(loaded_char3_index["num_entities"]),
            "num_terms": int(loaded_char3_index["num_terms"]),
            "matrix_shape": list(loaded_char3_index["bm25_matrix"].shape),
            "matrix_nnz": int(loaded_char3_index["bm25_matrix"].nnz),
            "k1": loaded_char3_index["k1"],
            "b": loaded_char3_index["b"],
        },
    },
    "usage_note": (
        "Load each joblib file once during traversal. "
        "Search returns entity_id values directly usable as KG entity node ids."
    ),
}

atomic_json_dump(bm25_meta, BM25_META_JSON_PATH)

print("Saved BM25 meta:", BM25_META_JSON_PATH)
print(json.dumps(bm25_meta, ensure_ascii=False, indent=2)[:3000])

Saved BM25 meta: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_bm25_indexes_meta.json
{
  "dataset": "hotpotqa",
  "created_at_utc": "2026-05-24T13:23:52.708771+00:00",
  "entity_id_mapping": "entity_id equals BM25 row index and KG entity node id",
  "indexes": {
    "entity_token_bm25": {
      "path": "/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_token_bm25.joblib",
      "format": "joblib",
      "index_type": "token_bm25",
      "num_entities": 327873,
      "num_terms": 124503,
      "matrix_shape": [
        327873,
        124503
      ],
      "matrix_nnz": 842075,
      "k1": 1.5,
      "b": 0.75
    },
    "entity_char3_bm25": {
      "path": "/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_char3_bm25.joblib",
      "format": "joblib",
      "index_type": "char3_bm25",
      "num_entities": 327873,
      "num_terms": 37385,
      "matrix_shape": [
        327873

In [14]:
#cell 14
# Update KG metadata with BM25 sidecar indexes.

with open(KG_META_JSON_PATH, "r", encoding="utf-8") as f:
    kg_meta = json.load(f)

kg_meta.setdefault("retrieval_indexes", {})

kg_meta["retrieval_indexes"]["entity_token_bm25"] = {
    "path": str(TOKEN_BM25_INDEX_PATH),
    "format": "joblib",
    "index_type": "token_bm25",
    "entity_id_mapping": "row index equals entity_id and KG entity node id",
    "metadata_source": str(BM25_META_JSON_PATH),
}

kg_meta["retrieval_indexes"]["entity_char3_bm25"] = {
    "path": str(CHAR3_BM25_INDEX_PATH),
    "format": "joblib",
    "index_type": "char3_bm25",
    "entity_id_mapping": "row index equals entity_id and KG entity node id",
    "metadata_source": str(BM25_META_JSON_PATH),
}

kg_meta["updated_at_utc"] = datetime.now(timezone.utc).isoformat()

atomic_json_dump(kg_meta, KG_META_JSON_PATH)

print("Updated KG metadata:", KG_META_JSON_PATH)
print("Registered retrieval indexes:")
print(json.dumps(kg_meta["retrieval_indexes"], ensure_ascii=False, indent=2))

Updated KG metadata: /content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_hetero_kg_meta.json
Registered retrieval indexes:
{
  "entity_token_bm25": {
    "path": "/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_token_bm25.joblib",
    "format": "joblib",
    "index_type": "token_bm25",
    "entity_id_mapping": "row index equals entity_id and KG entity node id",
    "metadata_source": "/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_bm25_indexes_meta.json"
  },
  "entity_char3_bm25": {
    "path": "/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_char3_bm25.joblib",
    "format": "joblib",
    "index_type": "char3_bm25",
    "entity_id_mapping": "row index equals entity_id and KG entity node id",
    "metadata_source": "/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/kg_construct/hotpotqa_entity_bm25_indexes_meta.json"
  }
}


In [15]:
#cell 15
# Final verification.

output_paths = [
    TOKEN_BM25_INDEX_PATH,
    CHAR3_BM25_INDEX_PATH,
    BM25_META_JSON_PATH,
    KG_META_JSON_PATH,
]

print("Output files:")
for path in output_paths:
    print(f"{path.name}: {round(file_size_mb(path), 2)} MB")

assert TOKEN_BM25_INDEX_PATH.exists()
assert CHAR3_BM25_INDEX_PATH.exists()
assert BM25_META_JSON_PATH.exists()
assert KG_META_JSON_PATH.exists()

assert loaded_token_index["num_entities"] == num_entities
assert loaded_char3_index["num_entities"] == num_entities

print("\nVerification passed.")
print("BM25 entity_id is directly usable as KG entity node id.")

Output files:
hotpotqa_entity_token_bm25.joblib: 7.73 MB
hotpotqa_entity_char3_bm25.joblib: 30.71 MB
hotpotqa_entity_bm25_indexes_meta.json: 0.0 MB
hotpotqa_hetero_kg_meta.json: 0.01 MB

Verification passed.
BM25 entity_id is directly usable as KG entity node id.


In [16]:
#cell 16
# Example retrieval function for traversal notebook.

def load_entity_bm25_indexes():
    token_index = joblib.load(TOKEN_BM25_INDEX_PATH)
    char3_index = joblib.load(CHAR3_BM25_INDEX_PATH)
    return token_index, char3_index


def retrieve_entity_candidates_by_bm25(
    query_entity,
    token_index,
    char3_index,
    top_k_token=4,
    top_k_char3=4,
):
    token_hits = search_entity_bm25(
        query=query_entity,
        index_payload=token_index,
        top_k=top_k_token,
    )

    char3_hits = search_entity_bm25(
        query=query_entity,
        index_payload=char3_index,
        top_k=top_k_char3,
    )

    merged = {}
    for hit in token_hits:
        entity_id = hit["entity_id"]
        merged.setdefault(entity_id, {
            "entity_id": entity_id,
            "entity": hit["entity"],
            "token_bm25_score": 0.0,
            "char3_bm25_score": 0.0,
        })
        merged[entity_id]["token_bm25_score"] = hit["score"]

    for hit in char3_hits:
        entity_id = hit["entity_id"]
        merged.setdefault(entity_id, {
            "entity_id": entity_id,
            "entity": hit["entity"],
            "token_bm25_score": 0.0,
            "char3_bm25_score": 0.0,
        })
        merged[entity_id]["char3_bm25_score"] = hit["score"]

    return list(merged.values())


token_index, char3_index = load_entity_bm25_indexes()

query_entity = "german language"

candidates = retrieve_entity_candidates_by_bm25(
    query_entity=query_entity,
    token_index=token_index,
    char3_index=char3_index,
    top_k_token=4,
    top_k_char3=4,
)

print(json.dumps(candidates, ensure_ascii=False, indent=2))

[
  {
    "entity_id": 63882,
    "entity": "german language",
    "token_bm25_score": 15.566742897033691,
    "char3_bm25_score": 77.62757873535156
  },
  {
    "entity_id": 64254,
    "entity": "german language learners",
    "token_bm25_score": 13.038372039794922,
    "char3_bm25_score": 61.25089645385742
  },
  {
    "entity_id": 290828,
    "entity": "german language countries",
    "token_bm25_score": 13.038372039794922,
    "char3_bm25_score": 0.0
  },
  {
    "entity_id": 258512,
    "entity": "german language expansion",
    "token_bm25_score": 13.038372039794922,
    "char3_bm25_score": 59.998756408691406
  },
  {
    "entity_id": 63843,
    "entity": "germanic language",
    "token_bm25_score": 0.0,
    "char3_bm25_score": 66.1351318359375
  }
]
